In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/colab_env/lib/python3.10/site-packages')

In [ ]:
from google.colab import userdata
from together import Together

# Initialize the Together client with the API key
client = Together(api_key=userdata.get('TOGETHER_API_KEY'))

In [ ]:
def get_completion(prompt, model="meta-llama/Meta-Llama-3-70B-Instruct-Turbo"):
    response = client.completions.create(
        model=model,
        prompt=prompt,
        temperature=0.0,  # Adjusting the temperature for better randomness control
    )
    return response.choices[0].text

In [ ]:
#print("Full Response Structure:", str(response.choices[0]))

def get_completion_from_messages(messages, model="meta-llama/Llama-3.3-70B-Instruct-Turbo", temperature=0.0):
    # Combine messages into a single prompt
    prompt = "\n".join([f"{msg['role']}: {msg['content']}" for msg in messages])

    response = client.completions.create(
        model=model,
        prompt=prompt,
        temperature=temperature,  # Adjusting the temperature for better randomness control
    )
    return response.choices[0].text

In [ ]:
messages = [
    {'role':'system', 'content':'You are a chatbot that provides concise and direct answers.'},
    {'role': 'user', 'content': 'Tell me a joke.'},
    {'role': 'assistant', 'content': 'Why did the chicken cross the road?'},
    {'role': 'user', 'content': 'you tell me.'}
]

In [ ]:
response = get_completion_from_messages(messages, temperature=0.0)
print(response)

In [ ]:
messages =  [
{'role':'system', 'content':'You are a chatbot that strictly provides direct responses without adding any extra information.'},
{'role':'user', 'content':'Hi, my name is Arman.'}  ]

In [ ]:
response = get_completion_from_messages(messages, temperature=0.0)
print(response)

In [ ]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot that provides concise and direct answers.'},
{'role':'user', 'content':'i cant remember my name, can you tell me What was my name?'}  ]
response = get_completion_from_messages(messages, temperature=0.0)
print(response)

In [ ]:
messages =  [
{'role':'system', 'content':'You are chatbot like chat-based models.'},
{'role':'user', 'content':'Hi, my name is Arman'},
{'role':'assistant', 'content': "Hi Arman! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=0.0)
print(response)

In [ ]:
# کتابخانه های مورد نیاز
import ipywidgets as widgets
from IPython.display import display, clear_output
from together import Together
from google.colab import userdata
import time  # ماژول تایم برای شبیه‌سازی زمان فکر کردن مدل

client = Together(api_key=userdata.get('TOGETHER_API_KEY'))

# نقش سیستم را تعریف کردیم
system_context = """
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collect the order, \
and then ask if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally, you collect the payment. \
Make sure to clarify all options, extras, and sizes to uniquely \
identify the item from the menu. \
The menu includes: \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""

# تابع گرفتن جواب از مدل
def get_completion_from_messages(messages, model="meta-llama/Llama-3.3-70B-Instruct-Turbo", temperature=0.7, max_tokens=150):
    # نقش سیستم را وارد می‌کنیم تا با مکالمه ادغام شود
    prompt = system_context + "\n\n"

    for message in messages:
        role = message['role']
        content = message['content']
        if role == 'user':
            prompt += f"Customer: {content}\n"
        elif role == 'assistant':
            prompt += f"Assistant: {content}\n"

    # اینجا به مدل میگیم که نوبت شماست
    prompt += "Assistant:"

    #درخواستمان را سمت مدل می‌فرستیم
    response = client.completions.create(
        model=model,
        prompt=prompt,
        temperature=temperature,  # Control response randomness
        max_tokens=max_tokens,    # Limit response length for conciseness
    )

    # برای اینکه خروجی فرم خوبی داشته باشد و جای خالی اول و آخر تکست را جدا کنیم
    assistant_response = response.choices[0].text.strip()

    # چک می‌کنیم فقط خروجی برای دستیار باشد و محتوای ورودی ما با آن ادغام نشود
    if "Customer:" in assistant_response:
        assistant_response = assistant_response.split("Customer:")[0].strip()

    return assistant_response

# متغیری برای ذخیره مکالمات تعریف می‌کنیم
context = []

# اجزای رابط گرافیکی
input_box = widgets.Textarea(
    placeholder='Type your message here...',
    layout=widgets.Layout(width='100%', height='60px'),
    style={'description_width': 'initial'}
)
output_box = widgets.Output(
    layout={'border': '1px solid black', 'padding': '10px', 'height': '400px', 'overflow_y': 'scroll'}
)

# تعریف دکمه‌ها برای رابط کاربری
submit_button = widgets.Button(description="Send")
reset_button = widgets.Button(description="Reset")

# تعریف دکمه ارسال برای ورودی کاربر
def on_submit(_):
    user_message = input_box.value.strip()
    input_box.value = ''

    if not user_message:
        return  # اگر پیام خالی بود، هیچ کاری انجام نده

    # پیام کاربر به لیست مکالمات اضافه می‌شود
    context.append({'role': 'user', 'content': user_message})

    # شبیه‌سازی زمان فکر کردن مدل
    with output_box:
        clear_output(wait=True) # پاک کردن محتویات باکس خروجی قبلی
        for message in context:
            print(f"\n{message['role'].capitalize()}: {message['content']}") # نمایش پیام های موجود
        print("\nAssistant is thinking...")

    time.sleep(1)

    # تابع تولید پاسخ
    response = get_completion_from_messages(context)

    #اضافه کردن پاسخ دستیار به متن مکالمه
    context.append({'role': 'assistant', 'content': response})

    # مکالمه کاربر و دستیار کامل نمایش داده میشه مدل آماده دریافت پیام جدید است
    with output_box:
        clear_output(wait=True)
        for message in context:
            print(f"\n{message['role'].capitalize()}: {message['content']}")
        print("\nAssistant is ready.")

# تابع پاک کردن مکالمه
def on_reset(_):
    global context
    context = []  # ریست کردن مکالمه
    # خروجی قبلی را پاک میکنیم
    with output_box:
        clear_output(wait=True)
        print("Conversation reset. Ready to start over.")

# با کلیلک روی دکمه تابع مربوطه اجرا می‌شود
submit_button.on_click(on_submit)
reset_button.on_click(on_reset)

display(output_box)
display(input_box)
display(submit_button, reset_button)


Output(layout=Layout(border='1px solid black', height='400px', overflow_y='scroll', padding='10px'))

Textarea(value='', layout=Layout(height='60px', width='100%'), placeholder='Type your message here...', style=…

Button(description='Send', style=ButtonStyle())

Button(description='Reset', style=ButtonStyle())